In [ ]:
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
#Accessing API MetObs to retrive wind speed data from Roskilde Lufthavn Feb25-Feb26
file_path_ws = DATA_DIR / "DMI_metObs_ws_etc.csv"
BASE_URL = "https://opendataapi.dmi.dk/v2/metObs/collections/observation/items"

parameters = ["wind_speed"]  #wind speed [m/s]: Latest 10 minutes' mean wind speed measured 10 m over terrain 
parameters = ["wind_speed","wind_max","wind_dir","wind_min"]  #MORE PARAMETERS 

# List to collect DataFrames
dfs = []
for col in parameters:
    # Parameters
    params = {
        "stationId": "06170",                     # your station (ROSKILDE LUFTHAVN)
        "parameterId": col,                       # parameter to retrieve
        "datetime": "2025-02-01T00:00:00Z/2026-02-28T23:59:59Z",
        "limit": 300000  ,                           # max records
    }
    # Send request
    response = requests.get(BASE_URL, params=params)
    # Check success
    if response.status_code == 200:
        data = response.json()
        # Each record is a GeoJSON feature
        features = data.get("features", [])
        # Parse all stationValue fields into a list of dicts
        records = []
        for f in features:
            props = f.get("properties", {})
            geom = f.get("geometry", {})
            coords = geom.get("coordinates", None)
            # Include geometry if you want
            record = {
                **props,
                "longitude": coords[0] if coords else None,
                "latitude": coords[1] if coords else None
            }
            records.append(record)
        # Convert to DataFrame and add to list if not empty
        if records:
            df_param = pd.DataFrame(records)
            dfs.append(df_param)
    else:
        print("Error", response.status_code, response.text)
# Stack all DataFrames on top of each other
if dfs:
    df_stacked = pd.concat(dfs, ignore_index=True)
else:
    print("No valid data returned for any parameter")
df_stacked.to_csv(file_path_ws)

In [ ]:
#Accessing API MetObs to retrive global radiation speed data from Sjaelsmark
file_path_rad = DATA_DIR / "DMI_metObs_rad_etc.csv"

BASE_URL = "https://opendataapi.dmi.dk/v2/metObs/collections/observation/items"

parameters = ["radia_glob"] #global irradiance [W/m^2]: Latest 10 minutes global radiation mean intensity
parameters = ["radia_glob","temp_dry"] #global irradiance [W/m^2]: Latest 10 minutes global radiation mean intensity

# List to collect DataFrames
dfs = []
for col in parameters:
    # Parameters
    params = {
        "stationId": "06188",                     # your station (SJAELSMARK)
        "parameterId": col,                       # parameter to retrieve
        "datetime": "2025-02-01T00:00:00Z/2026-02-28T23:59:59Z",
        "limit": 300000 ,                           # max records
    }
    # Send request
    response = requests.get(BASE_URL, params=params)
    # Check success
    if response.status_code == 200:
        data = response.json()
        # Each record is a GeoJSON feature
        features = data.get("features", [])
        # Parse all stationValue fields into a list of dicts
        records = []
        for f in features:
            props = f.get("properties", {})
            geom = f.get("geometry", {})
            coords = geom.get("coordinates", None)
            # Include geometry if you want
            record = {
                **props,
                "longitude": coords[0] if coords else None,
                "latitude": coords[1] if coords else None
            }
            records.append(record)
        # Convert to DataFrame and add to list if not empty
        if records:
            df_param = pd.DataFrame(records)
            dfs.append(df_param)
    else:
        print("Error", response.status_code, response.text)
# Stack all DataFrames on top of each other
if dfs:
    df_stacked = pd.concat(dfs, ignore_index=True)
else:
    print("No valid data returned for any parameter")
df_stacked.to_csv(file_path_rad)

,value,observed

In [ ]:
#Clean csv wind and irradiance data

def format_DMI_data(file_path,columnname):
    drop_columns = ["parameterId","created","stationId","longitude","latitude"]
    try:
        df = pd.read_csv(file_path,delimiter=",",decimal=".")
    except Exception as e:
        print(f"Error reading file: {e}")
        return
    
    df = df.drop(columns=drop_columns)

    df = df.rename(columns={"observed":"ts","value":columnname}) # rename columns

    df["ts"] = pd.to_datetime(df['ts']).dt.strftime('%Y-%m-%d %H:%M:%S') # setting date time and reformatting it 

    df = df.iloc[::-1].set_index("ts") # making time more formward instead of backward 

    df = df.drop(columns="Unnamed: 0") # drop earlier index column 
    
    output_path = file_path.parent / f"clean_{file_path.name}"
    
    df.to_csv(output_path) 

In [ ]:
def format_DMI_data_2(file_path,file_name):
    try:
        df = pd.read_csv(file_path,delimiter=",",decimal=".")
    except Exception as e:
        print(f"Error reading file: {e}")
        return
    
    df = df.rename(columns={"observed":"ts"}) # rename columns
    df["ts"] = pd.to_datetime(df['ts']) # setting date time


    df = df.pivot_table(
    index='ts', 
    columns="parameterId",
    values='value', 
    aggfunc='first')
    
    df.index = df.index.strftime('%Y-%m-%d %H:%M:%S')

    df.to_csv(file_name) 

In [ ]:
format_DMI_data_2(file_path_rad,DATA_DIR /"DMI_radiation.csv")

In [ ]:
format_DMI_data_2(file_path_ws,DATA_DIR /"DMI_windspeed.csv")